In [ ]:
from __future__ import annotations
import argparse
import time
from pathlib import Path
import ffmpeg
from loguru import logger

output_dir = "/Users/jcarpenter/Movies/converted/Movies"
source_dir = "/Users/jcarpenter/Downloads/movies"

def find_english_audio_stream(mkv_path: Path) -> int:
    """Return the first English, non-commentary audio stream's ffprobe index."""

    # ensures that we are only probing for audio streams and not video or subtitle streams
    probe = ffmpeg.probe(str(mkv_path), select_streams="a")
    for stream in probe.get("streams", []):
        if stream.get("codec_type") != "audio":
            continue

        tags = stream.get("tags", {})
        language = tags.get("language", "").lower()
        title = tags.get("title", "").lower()
        handler_name = tags.get("handler_name", "").lower()
        disposition = stream.get("disposition", {})
        is_commentary = disposition.get("commentary", 0) == 1
        is_commentary = (
            is_commentary
            or "commentary" in title
            or "commentary" in handler_name
        )

        if language in {"en", "eng"} and not is_commentary:
            return int(stream["index"])

    raise ValueError(
        f"No English, non-commentary audio stream found in {mkv_path}"
    )

def find_video_encoder(mkv_path: Path) -> str:
    """Return the matching encoder for an H.264 or HEVC video stream."""
    probe = ffmpeg.probe(
        str(mkv_path),
        select_streams="v:0",
        show_entries="stream=codec_name",
    )
    streams = probe.get("streams", [])
    codec_name = streams[0].get("codec_name", "").lower() if streams else ""

    if codec_name == "hevc":
        return "libx265"
    if codec_name == "h264":
        return "libx264"

    logger.warning(
        f"Unsupported source video codec '{codec_name}' in {mkv_path}; "
        "using libx264"
    )
    return "libx264"


def iter_mkv_files(source_dir: Path) -> Path:
    """Yield all .mkv files in the source directory."""
    for path in source_dir.iterdir():
        if path.is_file() and path.suffix.lower() == ".mkv":
            yield path


def format_duration(seconds: float) -> str:
    """Format a duration as hours, minutes, and seconds."""
    total_seconds = max(0, int(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"
    return f"{minutes:02d}:{seconds:02d}"


In [10]:
mkv_path = mkv_files[1]
mkv_path

PosixPath('/Users/jcarpenter/Downloads/movies/Wrong Turn (2003).mkv')

In [11]:
source_dir = Path(source_dir)
output_dir = Path(output_dir)
mkv_files = list(iter_mkv_files(source_dir))
output_path = output_dir / (mkv_path.stem + ".mp4")

In [12]:
audio_stream_index = find_english_audio_stream(mkv_path)
video_encoder = find_video_encoder(mkv_path)
duration_probe = ffmpeg.probe(
    str(mkv_path),
    select_streams="a",
    show_entries="format=duration",
)
duration = float(duration_probe["format"]["duration"])

In [15]:
srt_path = mkv_path.with_suffix(".srt")
if srt_path.exists():
    logger.info(f"Found subtitle: {srt_path}")
    input_kwargs = {
        "filename": str(mkv_path),
        "s": str(srt_path),
        "analyzeduration": "100M",
        "probesize": "100M",
        "fflags": "+genpts"
    }
else:
    input_kwargs = {
        "filename": str(mkv_path),
        "analyzeduration": "100M",
        "probesize": "100M",
        "fflags": "+genpts"
    }

In [19]:
input_kwargs['probesize'] = '500M'
preset = "medium"
crf = 23

In [20]:
input_stream = ffmpeg.input(**input_kwargs)
process = (
    ffmpeg.output(
        input_stream.video,
        input_stream[str(audio_stream_index)],
        str(output_path),
        vcodec=video_encoder,
        acodec="aac",
        af="aresample=async=1:first_pts=0",
        ac=2,
        audio_bitrate="320k",
        preset=preset,
        crf=crf,
        movflags="+faststart",
    )
    .global_args("-progress", "pipe:1", "-nostats")
    .run_async(pipe_stdout=True, overwrite_output=True)
)

ffmpeg version 9.0.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.1.1.101)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg-full/9.0.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-libplacebo --enable-libqrencode --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-l